In [6]:
#Abrimos una sesión de spark
from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("MySpark")
    .master("local[8]")                # Usar exactamente 8 cores
    .config("spark.sql.shuffle.partitions", "6")   # Particiones post-shuffle
    .config("spark.driver.memory", "4g")           # RAM del driver
    .config("spark.sql.adaptive.enabled", "true")  # Adaptive Query Execution
    .getOrCreate()
        )
# Verificar versión
print(spark.version)            # 3.x.x
print(spark.sparkContext.uiWebUrl)  # http://localhost:4040

3.5.0
http://d17fa7338063:4040


In [7]:
sc = spark.sparkContext

print(sc.appName)           # "MySpark"
print(sc.master)            # "local[4]"
print(sc.defaultParallelism) # Número de cores
print(sc.version)           # Versión de Spark

# ¡NUNCA hagas esto en producción!
# spark.stop()  # Mata toda la sesión

MySpark
local[8]
8
3.5.0


In [8]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, LongType,
    DoubleType, FloatType,
    BooleanType, DateType, TimestampType,
    ArrayType, MapType
)

# ── Esquema básico ─────────────────────────────────────
schema_precios = StructType([
    StructField("Fecha",       DateType(),      nullable=True),
    StructField("Presentacion",      StringType(),    nullable=False),
    StructField("Origen",     StringType(),    nullable=False),
    StructField("Destino",   StringType(),    nullable=True),
    StructField("Precio Min",   DoubleType(),   nullable=True),
    StructField("Precio Max",   DoubleType(),   nullable=True),
    StructField("Precio Frec",   DoubleType(),   nullable=True),
    StructField("Obs",   StringType(),   nullable=True)
])

df_precio = spark.read \
    .option("header", "true") \
    .option("sep",",")  \
    .option("quote", '"') \
    .option("escape", '"') \
    .schema(schema_precios) \
    .option("nullValue", "N/A") \
    .option("encoding",     "UTF-8") \
    .option("nanValue",     "nan") \
    .csv("data/sniim_precio_aguacate_3.csv")

df_precio.printSchema()

root
 |-- Fecha: date (nullable = true)
 |-- Presentacion: string (nullable = true)
 |-- Origen: string (nullable = true)
 |-- Destino: string (nullable = true)
 |-- Precio Min: double (nullable = true)
 |-- Precio Max: double (nullable = true)
 |-- Precio Frec: double (nullable = true)
 |-- Obs: string (nullable = true)



In [9]:
print(f'--- Se van a analizar {df_precio.count()} filas')

df_precio.show(10)

--- Se van a analizar 717114 filas
+----------+------------+---------+--------------------+----------+----------+-----------+---+
|     Fecha|Presentacion|   Origen|             Destino|Precio Min|Precio Max|Precio Frec|Obs|
+----------+------------+---------+--------------------+----------+----------+-----------+---+
|2001-01-02|   Kilogramo|Michoacán|Aguascalientes: C...|       9.0|       9.5|        9.5| --|
|2001-01-02|   Kilogramo|Michoacán|Aguascalientes: C...|       9.0|       9.5|        9.5| --|
|2001-01-02|   Kilogramo|Michoacán|Aguascalientes: C...|       9.0|       9.5|        9.5| --|
|2001-01-03|   Kilogramo|Michoacán|Aguascalientes: C...|       9.0|       9.5|        9.5| --|
|2001-01-03|   Kilogramo|Michoacán|Aguascalientes: C...|       9.0|       9.5|        9.5| --|
|2001-01-03|   Kilogramo|Michoacán|Aguascalientes: C...|       9.0|       9.5|        9.5| --|
|2001-01-04|   Kilogramo|Michoacán|Aguascalientes: C...|       9.0|       9.5|        9.5| --|
|2001-01-04|   

In [10]:
#Comenzamos con la limpieza de la Base

#Eliminamos los duplicados exactos, sniim solo hace un reporte por fecha, destino, origen y presentación
df_sin_dupes = df_precio.dropDuplicates() 
sin_dup = df_sin_dupes.count()
total = df_precio.count()
print(f'De {total} filas, {total - sin_dup} son duplicadas, al final quedan {sin_dup} registros')

#Calculamos el porcentaje de retención
print(f'Los datos no duplicados representan {round((sin_dup/total)*100, 2)} % del total de los registros')

De 717114 filas, 478076 son duplicadas, al final quedan 239038 registros
Los datos no duplicados representan 33.33 % del total de los registros


In [11]:
#Comenzamos con la limpieza de los datos

from pyspark.sql import functions as F
#-------Revisamos los nulls por columna--------------------
print('Conteo de datos nulos por columna')

nulos_por_col = df_sin_dupes.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_sin_dupes.columns
])
nulos_por_col.show()


# ── Porcentaje de nulls ────────────────────────────────

print('Porcentaje de datos nulos por columna')

total = df_sin_dupes.count()
pct_nulos = df_sin_dupes.select([
    F.round(
        F.count(F.when(F.col(c).isNull(), c)) / total * 100, 2
    ).alias(c)
    for c in df_sin_dupes.columns
])
pct_nulos.show()

Conteo de datos nulos por columna
+-----+------------+------+-------+----------+----------+-----------+------+
|Fecha|Presentacion|Origen|Destino|Precio Min|Precio Max|Precio Frec|   Obs|
+-----+------------+------+-------+----------+----------+-----------+------+
|    0|           0|     0|      0|     10650|      7360|       5815|174943|
+-----+------------+------+-------+----------+----------+-----------+------+

Porcentaje de datos nulos por columna
+-----+------------+------+-------+----------+----------+-----------+-----+
|Fecha|Presentacion|Origen|Destino|Precio Min|Precio Max|Precio Frec|  Obs|
+-----+------------+------+-------+----------+----------+-----------+-----+
|  0.0|         0.0|   0.0|    0.0|      4.46|      3.08|       2.43|73.19|
+-----+------------+------+-------+----------+----------+-----------+-----+



In [12]:
# Observamos qué está pasando con esos datos nulos
# En el caso de Obs, convendría descartar la columna ya que el 73% de sus datos son nulos

from pyspark.sql.functions import col

obs = df_sin_dupes.select('Obs').dropDuplicates()#Valores Únicos de las observaciones

conteo_obs_tot = df_sin_dupes.filter(col("Obs").isNotNull())

print(f'De {conteo_obs_tot.count()} observaciones no nulas, {obs.count()} son valores únicos')

obs.show()

#Para conocer la frecuencia de cada observación
print('Frecuencia de las observaciones')
# Agrupamos, contamos y ordenamos de mayor a menor (descendente)
conteo_obs_ordenado = df_sin_dupes.groupBy("Obs") \
                        .count() \
                        .orderBy(col("count").desc())

# Mostramos los resultados (ej. los top 20 más frecuentes)
conteo_obs_ordenado.show(truncate=False)

De 64095 observaciones no nulas, 1103 son valores únicos
+--------------------+
|                 Obs|
+--------------------+
|              250.00|
|              160.00|
|           flor loca|
|el precio de este...|
|este precio corre...|
| Producción Reciente|
|      SIN CLASIFICAR|
|Los precios del p...|
|alza por poca oferta|
|subio el precio p...|
|          FLOR  LOCA|
|           Flor Loca|
|              265.00|
|        EXTRA  EXTRA|
|         SUPER EXTRA|
|             Primera|
|    FLORACION N UEVA|
|EXTRA  MIN$13.00M...|
|      CALIDAD  EXTRA|
|    CALIDAD    EXTRA|
+--------------------+
only showing top 20 rows

Frecuencia de las observaciones
+---------------------+------+
|Obs                  |count |
+---------------------+------+
|NULL                 |174943|
|--                   |45510 |
|EXTRA                |2030  |
|CALIDAD EXTRA        |1871  |
|CAJA DE MADERA       |1665  |
|FLOR LOCA            |846   |
|Precio según tamaño. |440   |
|280.00               |

In [13]:
from pyspark.sql.functions import col
from pyspark.sql import functions as F

# Observamos ahora qué pasa con los null de los precios
df_nulos = df_sin_dupes.filter(col("Precio Frec").isNull())
df_nulos.show(5)

df_nulos = df_sin_dupes.filter(col("Precio Min").isNull())
df_nulos.show(5)

df_nulos = df_sin_dupes.filter(col("Precio Max").isNull())
df_nulos.show(5)

+----------+--------------+---------+--------------------+----------+----------+-----------+----+
|     Fecha|  Presentacion|   Origen|             Destino|Precio Min|Precio Max|Precio Frec| Obs|
+----------+--------------+---------+--------------------+----------+----------+-----------+----+
|2011-07-15|Caja de 24 kg.|Michoacán|Hidalgo: Central ...|      NULL|      NULL|       NULL|NULL|
|2011-08-03|Caja de 24 kg.|Michoacán|Hidalgo: Central ...|      NULL|      NULL|       NULL|NULL|
|2011-08-18|Caja de 24 kg.|Michoacán|Hidalgo: Central ...|      NULL|      NULL|       NULL|NULL|
|2011-08-22|Caja de 24 kg.|Michoacán|Hidalgo: Central ...|      NULL|      NULL|       NULL|NULL|
|2011-08-29|Caja de 24 kg.|Michoacán|Hidalgo: Central ...|      NULL|      NULL|       NULL|NULL|
+----------+--------------+---------+--------------------+----------+----------+-----------+----+
only showing top 5 rows

+----------+--------------+----------------+--------------------+----------+----------+------

In [14]:
#Observamos que cuando no hay precio frec, por lo general no están los otros datos
#Cuando no hay precio min, es porque los valores están recorridos
#Cuando falta el precio max es porque solo falta ese dato

from pyspark.sql.functions import col, when
#Para Corregir el Precio Min...

# 1. Definimos la condición doble:
# - Que 'Precio Min' sea nulo
# - Y que 'Obs' se pueda convertir a un número (lo que confirmaría que es un precio desplazado)
condicion = col("Precio Min").isNull() & col("Obs").cast("double").isNotNull()

# 2. Creamos columnas temporales con los valores recorridos a la izquierda
df_corregido = df_sin_dupes \
    .withColumn("Precio_Min_temp", when(condicion, col("Precio Max")).otherwise(col("Precio Min"))) \
    .withColumn("Precio_Max_temp", when(condicion, col("Precio Frec")).otherwise(col("Precio Max"))) \
    .withColumn("Precio_Frec_temp", when(condicion, col("Obs")).otherwise(col("Precio Frec"))) \
    .withColumn("Obs_temp", when(condicion, None).otherwise(col("Obs")))

# 3. Borramos las columnas viejas defectuosas
df_corregido = df_corregido.drop("Precio Min", "Precio Max", "Precio Frec", "Obs")

# 4. Renombramos las columnas temporales para que tengan el nombre original
df_final = df_corregido \
    .withColumnRenamed("Precio_Min_temp", "Precio Min") \
    .withColumnRenamed("Precio_Max_temp", "Precio Max") \
    .withColumnRenamed("Precio_Frec_temp", "Precio Frec") \
    .withColumnRenamed("Obs_temp", "Obs")

# Verificamos el resultado

# Filtramos donde la conversión a 'double' sea exitosa (es decir, no sea nula)
df_obs_numericas = df_final.filter(col("Obs").cast("double").isNotNull())

# Mostramos el resultado
df_obs_numericas.show()

+----------+--------------+----------------+--------------------+----------+----------+-----------+---+
|     Fecha|  Presentacion|          Origen|             Destino|Precio Min|Precio Max|Precio Frec|Obs|
+----------+--------------+----------------+--------------------+----------+----------+-----------+---+
|2007-08-09|Caja de 15 kg.|       Michoacán|Morelos: Central ...|     450.0|     470.0|      460.0|130|
|2007-10-18|Caja de 13 kg.|       Michoacán|Morelos: Central ...|     250.0|     260.0|      250.0|120|
|2010-07-13|Caja de 10 kg.|       Michoacán|Nayarit: Mercado ...|     300.0|     310.0|      310.0|310|
|2009-07-27|Caja de 20 kg.|       Michoacán|"\"Guanajuato: Me...|     640.0|     660.0|      660.0|  0|
|2009-11-30|Caja de 18 kg.|       Michoacán|Veracruz: Central...|     230.0|     235.0|      235.0|  5|
|2007-05-23|Caja de 18 kg.|Distrito Federal|Veracruz: Mercado...|     200.0|     250.0|      250.0|250|
|2010-03-01|     Kilogramo|       Michoacán|Sonora: Central d...

In [15]:
#Hacemos un análisis de texto para saber qué se comenta principalmente en las Observaciones cada año

from pyspark.sql.functions import col, year, split, explode, lower, regexp_replace, count, row_number
from pyspark.sql.window import Window

df_reducido = df_final.select("Fecha", "Obs").filter(col("Obs").isNotNull())

# Divide los datos en 50 bloques pequeños para procesarlos poco a poco
df_reducido = df_reducido.repartition(50)

# Ahora sí, aplica la lógica anterior sobre df_reducido en lugar de df_final
df_palabras = df_reducido \
    .withColumn("Anio", year(col("Fecha"))) \
    .withColumn("Obs_Limpia", lower(regexp_replace(col("Obs"), "[^a-zA-Z0-9áéíóúñÑ\\s]", ""))) \
    .withColumn("Palabra", explode(split(col("Obs_Limpia"), "\\s+"))) \
    .filter(col("Palabra") != "")

# Lista de palabras a ignorar
stop_words = ["de", "la", "el", "en", "y", "a", "los", "las", "un", "una", "con", "sin"]

# Agregamos esta línea justo después del explode()
df_palabras = df_palabras.filter(~col("Palabra").isin(stop_words))

# PASO 3: Contar frecuencias agrupando por Año y por Palabra
df_conteo = df_palabras.groupBy("Anio", "Palabra").agg(count("*").alias("Frecuencia"))

# PASO 4: Encontrar la palabra top por año usando Window Functions
# Creamos una "ventana" separada por año, ordenada por frecuencia de mayor a menor
ventana_anio = Window.partitionBy("Anio").orderBy(col("Frecuencia").desc())

# Asignamos un número de fila (Ranking) a cada palabra dentro de su año
df_top_por_anio = df_conteo.withColumn("Ranking", row_number().over(ventana_anio)) \
                           .filter(col("Ranking") == 1) \
                           .drop("Ranking") \
                           .orderBy(col("Anio").desc()) # Ordenamos el resultado final por año

# Mostramos el resultado
df_top_por_anio.show(27)

+----+---------+----------+
|Anio|  Palabra|Frecuencia|
+----+---------+----------+
|2026|temporada|         1|
|2024|     loca|         6|
|2023|     flor|        55|
|2022|    según|         4|
|2021|    extra|       243|
|2020|    extra|       201|
|2019|    extra|       210|
|2018|   tamaño|       194|
|2017|   precio|       126|
|2016|   precio|        82|
|2015|     flor|        79|
|2014|  calidad|       285|
|2013|  calidad|       224|
|2012|  calidad|       332|
|2011|     flor|       207|
|2010|  calidad|        62|
|2009|     caja|       323|
|2008|     caja|       447|
|2007|     caja|       365|
|2006|    extra|       465|
|2005|    extra|      1219|
|2004|    extra|      1012|
|2003|    extra|       381|
|2002|floracion|        83|
|2001|     loca|        24|
+----+---------+----------+



In [17]:
#No parece haber nada interesante en las observaciones, asi que eliminamos la columna

df_final = df_final.drop('Obs')

# Eliminamos los valores nulos
df_limpio = df_final.dropna()

df_limpio.show()

+----------+------------+---------+--------------------+----------+----------+-----------+
|     Fecha|Presentacion|   Origen|             Destino|Precio Min|Precio Max|Precio Frec|
+----------+------------+---------+--------------------+----------+----------+-----------+
|2001-01-10|   Kilogramo|Michoacán|Aguascalientes: C...|       9.0|       9.5|        9.5|
|2001-01-15|   Kilogramo|Michoacán|Aguascalientes: C...|      10.0|      11.0|       10.5|
|2001-01-17|   Kilogramo|Michoacán|Aguascalientes: C...|      11.0|      12.0|       11.0|
|2001-01-30|   Kilogramo|Michoacán|Aguascalientes: C...|      11.5|      12.0|       12.0|
|2001-02-09|   Kilogramo|Michoacán|Aguascalientes: C...|      11.5|      12.0|       12.0|
|2001-02-27|   Kilogramo|Michoacán|Aguascalientes: C...|      11.5|      12.0|       12.0|
|2001-03-15|   Kilogramo|Michoacán|Aguascalientes: C...|      11.5|      12.0|       12.0|
|2001-03-16|   Kilogramo|Michoacán|Aguascalientes: C...|      11.5|      12.0|       12.0|

In [18]:
from pyspark.sql import functions as F

#Para finalizar la limpieza, observamos cuántos null pudimos reducir y cuántos registros vamos a eliminar
#-------Revisamos los nulls por columna--------------------
print('Conteo de datos nulos por columna')

nulos_por_col = df_limpio.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_limpio.columns
])
nulos_por_col.show()


# ── Porcentaje de nulls ────────────────────────────────

print('Porcentaje de datos nulos por columna')

total = df_limpio.count()
pct_nulos = df_limpio.select([
    F.round(
        F.count(F.when(F.col(c).isNull(), c)) / total * 100, 2
    ).alias(c)
    for c in df_limpio.columns
])
pct_nulos.show()

total =  df_sin_dupes.count()
limpio = df_limpio.count()
resto = total - limpio

pct_ret = round(limpio / total * 100, 2) 

print(f'De {total} datos no duplicados, se extrajeron {resto} y quedaron {limpio} registros')
print(f'El porcentaje de retención es de: {pct_ret} % del total de registros')

Conteo de datos nulos por columna
+-----+------------+------+-------+----------+----------+-----------+
|Fecha|Presentacion|Origen|Destino|Precio Min|Precio Max|Precio Frec|
+-----+------------+------+-------+----------+----------+-----------+
|    0|           0|     0|      0|         0|         0|          0|
+-----+------------+------+-------+----------+----------+-----------+

Porcentaje de datos nulos por columna
+-----+------------+------+-------+----------+----------+-----------+
|Fecha|Presentacion|Origen|Destino|Precio Min|Precio Max|Precio Frec|
+-----+------------+------+-------+----------+----------+-----------+
|  0.0|         0.0|   0.0|    0.0|       0.0|       0.0|        0.0|
+-----+------------+------+-------+----------+----------+-----------+

De 239038 datos no duplicados, se extrajeron 7360 y quedaron 231678 registros
El porcentaje de retención es de: 96.92 % del total de registros


In [19]:
#Como el precio varia según la presentación, necesitamos una columna del precio por kilo

#Primero Analizamos Los tipos de presentaciones que hay
presentaciones = df_limpio.select('Presentacion').dropDuplicates()
presentaciones.show(35)
print(f'Hay {presentaciones.count()} tipos de presentaciones')

+------------------+
|      Presentacion|
+------------------+
|    Caja de 18 kg.|
| Arpilla de 10 kg.|
|    Caja de 20 kg.|
|     Caja de 8 kg.|
|    Caja de 16 kg.|
|    Caja de 15 kg.|
|     Caja de 9 kg.|
|    Caja de 19 kg.|
|     Caja de 7 kg.|
|    Caja de 25 kg.|
|         Kilogramo|
|    Caja de 12 kg.|
|    Caja de 24 kg.|
|    Caja de 10 kg.|
|    Caja de 17 kg.|
| Arpilla de 24 kg.|
|    Caja de 21 kg.|
|    Caja de 14 kg.|
|    Caja de 13 kg.|
| Arpilla de 20 kg.|
|     Caja de 6 kg.|
|    Caja de 30 kg.|
| Arpilla de 13 kg.|
|   Cesta de 10 kg.|
|            Manojo|
|    Caja de 23 kg.|
|    Caja de 11 kg.|
|    Reja de 18 kg.|
|Manojo de 10 pzas.|
|    Reja de 20 kg.|
|     Caja de 5 kg.|
|    Caja de 22 kg.|
|    Caja de 26 kg.|
|  Manojo de 10 kg.|
|    Caja de 28 kg.|
+------------------+

Hay 35 tipos de presentaciones


In [20]:
conteo_tipos = df_limpio.groupBy("Presentacion").count()

# Mostramos el resultado
conteo_tipos.show(35)

+------------------+-----+
|      Presentacion|count|
+------------------+-----+
|    Caja de 18 kg.|15901|
| Arpilla de 10 kg.|   46|
|    Caja de 20 kg.|22068|
|     Caja de 8 kg.|  109|
|    Caja de 22 kg.|   41|
| Arpilla de 20 kg.|    4|
|    Caja de 16 kg.| 3591|
|    Caja de 15 kg.| 7911|
|     Caja de 9 kg.|21547|
|    Caja de 19 kg.| 5104|
|     Caja de 7 kg.|  212|
|     Caja de 6 kg.|  694|
|    Caja de 30 kg.|  178|
|    Caja de 25 kg.| 2751|
|   Cesta de 10 kg.|    8|
|            Manojo|    1|
|         Kilogramo|74612|
|    Caja de 12 kg.| 6023|
|    Caja de 24 kg.| 4869|
|    Caja de 10 kg.|50775|
|    Caja de 17 kg.| 2629|
| Arpilla de 24 kg.|    4|
|    Caja de 23 kg.|  112|
|    Caja de 11 kg.|  198|
|    Caja de 21 kg.|  129|
|    Caja de 14 kg.| 5283|
|    Caja de 13 kg.| 6646|
|    Reja de 18 kg.|  209|
|    Caja de 26 kg.|   16|
|Manojo de 10 pzas.|    1|
|  Manojo de 10 kg.|    2|
|    Reja de 20 kg.|    1|
|    Caja de 28 kg.|    1|
| Arpilla de 13 kg.|    1|
|

In [21]:
from pyspark.sql.functions import col, when, regexp_extract

# Creamos una columna que nos diga cuántos kilos trae la presentación
df_con_kilos = df_limpio.withColumn(
    "Cantidad_Kilos",
    when(col("Presentacion") == "Kilogramo", 1)
    .when(col("Presentacion") == "Manojo", 0.7)              # Excepción 1
    .when(col("Presentacion") == "Manojo de 10 pzas.", 3)  # Excepción 2
    .otherwise(
        # Si no es ninguna de las excepciones de arriba, busca el número
        regexp_extract(col("Presentacion"), r"(\d+)", 1).cast("int")
    )
)

# 2. Calculamos el Precio (con la validación para evitar dividir entre cero)
df_aumentado = df_con_kilos.withColumn(
    "Precio Kg",
    when(col("Cantidad_Kilos") > 0, col("Precio Frec") / col("Cantidad_Kilos")).otherwise(None)
)

# Verificamos
df_aumentado.select("Presentacion", "Cantidad_Kilos", "Precio Frec", "Precio Kg").show(truncate=False)

+------------+--------------+-----------+---------+
|Presentacion|Cantidad_Kilos|Precio Frec|Precio Kg|
+------------+--------------+-----------+---------+
|Kilogramo   |1.0           |9.5        |9.5      |
|Kilogramo   |1.0           |10.5       |10.5     |
|Kilogramo   |1.0           |11.0       |11.0     |
|Kilogramo   |1.0           |12.0       |12.0     |
|Kilogramo   |1.0           |12.0       |12.0     |
|Kilogramo   |1.0           |12.0       |12.0     |
|Kilogramo   |1.0           |12.0       |12.0     |
|Kilogramo   |1.0           |12.0       |12.0     |
|Kilogramo   |1.0           |13.0       |13.0     |
|Kilogramo   |1.0           |13.0       |13.0     |
|Kilogramo   |1.0           |13.0       |13.0     |
|Kilogramo   |1.0           |13.0       |13.0     |
|Kilogramo   |1.0           |12.0       |12.0     |
|Kilogramo   |1.0           |12.0       |12.0     |
|Kilogramo   |1.0           |12.0       |12.0     |
|Kilogramo   |1.0           |15.0       |15.0     |
|Kilogramo  

In [22]:
#-------Revisamos los nulls por columna--------------------
print('Conteo de datos nulos por columna')

nulos_por_col = df_aumentado.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_aumentado.columns
])
nulos_por_col.show()

Conteo de datos nulos por columna
+-----+------------+------+-------+----------+----------+-----------+--------------+---------+
|Fecha|Presentacion|Origen|Destino|Precio Min|Precio Max|Precio Frec|Cantidad_Kilos|Precio Kg|
+-----+------------+------+-------+----------+----------+-----------+--------------+---------+
|    0|           0|     0|      0|         0|         0|          0|             0|        0|
+-----+------------+------+-------+----------+----------+-----------+--------------+---------+



In [23]:
#Ahora que tenemos limpio nuestro df, procedemos a revisar los outliers o datos atipicos

#Revisamos los percentiles, para ver si hay algo raro 
df_aumentado.select('Precio Min', 'Precio Max', 'Precio Frec', 'Cantidad_kilos', 'Precio kg').summary().show()

+-------+------------------+------------------+-----------------+-----------------+------------------+
|summary|        Precio Min|        Precio Max|      Precio Frec|   Cantidad_kilos|         Precio kg|
+-------+------------------+------------------+-----------------+-----------------+------------------+
|  count|            231678|            231678|           231678|           231678|            231678|
|   mean| 257.3159678950964|275.08295630141845|265.8649091411343|9.777055654831274|31.290430821856948|
| stddev|219.57719116254776|233.87347786712988|226.7740992131166|7.207039842759588|19.474209601597977|
|    min|               4.0|               4.8|             10.0|              0.7|0.8871428571428571|
|    25%|              46.0|              50.0|             48.0|              1.0|              16.5|
|    50%|             225.0|             240.0|            230.0|             10.0|26.153846153846153|
|    75%|             390.0|             415.0|            400.0|        

In [24]:
#Agregamos columnas de anio y mes para una mejor lectura

from pyspark.sql.functions import col, year, month

# Agregamos las dos columnas encadenando withColumn
df_con_fechas = df_aumentado.withColumn("anio", year(col("Fecha"))) \
                  .withColumn("mes", month(col("Fecha")))

# Mostramos el resultado para verificar
df_con_fechas.show(5)

+----------+------------+---------+--------------------+----------+----------+-----------+--------------+---------+----+---+
|     Fecha|Presentacion|   Origen|             Destino|Precio Min|Precio Max|Precio Frec|Cantidad_Kilos|Precio Kg|anio|mes|
+----------+------------+---------+--------------------+----------+----------+-----------+--------------+---------+----+---+
|2001-01-10|   Kilogramo|Michoacán|Aguascalientes: C...|       9.0|       9.5|        9.5|           1.0|      9.5|2001|  1|
|2001-01-15|   Kilogramo|Michoacán|Aguascalientes: C...|      10.0|      11.0|       10.5|           1.0|     10.5|2001|  1|
|2001-01-17|   Kilogramo|Michoacán|Aguascalientes: C...|      11.0|      12.0|       11.0|           1.0|     11.0|2001|  1|
|2001-01-30|   Kilogramo|Michoacán|Aguascalientes: C...|      11.5|      12.0|       12.0|           1.0|     12.0|2001|  1|
|2001-02-09|   Kilogramo|Michoacán|Aguascalientes: C...|      11.5|      12.0|       12.0|           1.0|     12.0|2001|  2|


In [29]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, avg, stddev, abs

# 1. Definimos una ventana que ordene por fecha y mire los 90 días anteriores
# Nota: Asumimos que tus datos ya no tienen duplicados y están por ruta
ventana_90_dias = Window.partitionBy("Origen", "Destino").orderBy("Fecha").rowsBetween(-90, -1)

# 2. Calculamos la media y desviación estándar de ESOS 30 días específicos
df_tendencia = df_con_fechas.withColumn("Media_Movil", avg("Precio Kg").over(ventana_90_dias)) \
                 .withColumn("Desv_Movil", stddev("Precio Kg").over(ventana_90_dias))

# 2. Calculamos el Z-Score dinámico para cada fila (cuántas desviaciones se aleja de la media)
df_calibracion = df_tendencia.withColumn(
    "Z_Dinamico", 
    abs((col("Precio Kg") - col("Media_Movil")) / col("Desv_Movil"))
)

# 3. Creamos un resumen probando diferentes niveles de "estrictez"
resumen_umbrales = df_calibracion.select(
    count(when(col("Z_Dinamico") > 3, True)).alias("Umbral_3_Desv"),
    count(when(col("Z_Dinamico") > 4, True)).alias("Umbral_4_Desv"),
    count(when(col("Z_Dinamico") > 5, True)).alias("Umbral_5_Desv")
)

# 4. Mostrar la comparativa
resumen_umbrales.show()

+-------------+-------------+-------------+
|Umbral_3_Desv|Umbral_4_Desv|Umbral_5_Desv|
+-------------+-------------+-------------+
|         9458|         2809|         1081|
+-------------+-------------+-------------+



In [32]:
from pyspark.sql.functions import col

# Conservar solo los datos limpios (Todo lo que esté dentro de las 5 desviaciones)
# Usamos <= 5 para quedarnos con lo normal
df_limpio = df_calibracion.filter(col("Z_Dinamico") <= 5)

# los datos anomalos
df_errores = df_calibracion.filter(col("Z_Dinamico") > 5)

# Borramos la columna Z_Dinamico del df_limpio porque ya cumplió su función
df_limpio = df_limpio.drop("Z_Dinamico", "Media_Movil", "Desv_Movil")

n_errores = df_errores.count()
n_totales = df_con_fechas.count()

pct_outliers = (n_errores/n_totales) * 100

print(f"Total de registros listos para analizar: {df_limpio.count()}")
print(f'Total de outliers en el dataset limpio: {n_errores}')
print(f'El poreccntaje de datos que se eliminaron por ser anomalias (z-score > 5) es de: {round(pct_outliers, 3)} % del total')

Total de registros listos para analizar: 227754
Total de outliers en el dataset limpio: 1081
El poreccntaje de datos que se eliminaron por ser anomalias (z-score > 5) es de: 0.467 % del total


In [45]:
df_limpio.select("Destino").distinct().show(truncate=False)

total_destinos = df_limpio.select("Destino").distinct().count()
print(f"Total de destinos únicos: {total_destinos}")

+---------------------------------------------------------------+
|Destino                                                        |
+---------------------------------------------------------------+
|Chiapas: Central de Abasto de Tuxtla Gutiérrez                 |
|Colima: Centros de distribución de Colima                      |
|Guerrero: Central de Abastos de Acapulco                       |
|Jalisco: Mercado de Abasto de Guadalajara                      |
|"\"Zacatecas: Tianguis \"\"Francisco García Salinas\"\"\""     |
|Guanajuato: Central de Abasto de León                          |
|Yucatán: Centro Mayorista Oxkutzcab                            |
|Aguascalientes: Centro Comercial Agropecuario de Aguascalientes|
|Puebla: Central de Abasto de Puebla                            |
|"\"Campeche: Mercado \"\"Pedro Sáinz de Baranda\"\""           |
|"\"Yucatán: Mercado \"\"Casa del Pueblo\"\"\""                 |
|Veracruz: Mercado de San José de Xalapa                        |
|Chihuahua

In [50]:
from pyspark.sql.functions import col, split, trim, regexp_replace

# 1. Limpieza: Quitamos todas las comillas (") y diagonales invertidas (\)
# Usamos expresiones regulares para buscar esos símbolos y reemplazarlos por "nada"
df_destinos = df_limpio.withColumn("Destino_Limpio", regexp_replace(col("Destino"), '["\\\\]', ''))

# 2. Dividimos el texto usando los dos puntos (:) como separador
# Esto crea una lista o "arreglo" temporal con dos elementos por cada fila
columna_dividida = split(col("Destino_Limpio"), ":")

# 3. Creamos las dos nuevas columnas extrayendo los elementos de esa lista temporal
# getItem(0) saca la primera parte (el estado)
# getItem(1) saca la segunda parte (el mercado)
# Usamos trim() para eliminar cualquier espacio en blanco sobrante al principio o al final
df_destinos_final = df_destinos.withColumn("estado_destino", trim(columna_dividida.getItem(0))) \
                    .withColumn("centro_distribucion", trim(columna_dividida.getItem(1)))

# 4. (Opcional) Borramos la columna temporal que usamos para limpiar
df_destinos_final = df_destinos_final.drop("Destino_Limpio")

# Vemos el resultado comparando la original con las dos nuevas
print('DataFrame con el destino extraido')
df_destinos_final.select("Destino", "estado_destino", "centro_distribucion").show(20, truncate=False)

DataFrame con el destino extraido
+-------------------------------------+--------------+-----------------------+
|Destino                              |estado_destino|centro_distribucion    |
+-------------------------------------+--------------+-----------------------+
|Quintana Roo: Módulo de Abasto Cancún|Quintana Roo  |Módulo de Abasto Cancún|
|Quintana Roo: Módulo de Abasto Cancún|Quintana Roo  |Módulo de Abasto Cancún|
|Quintana Roo: Módulo de Abasto Cancún|Quintana Roo  |Módulo de Abasto Cancún|
|Quintana Roo: Módulo de Abasto Cancún|Quintana Roo  |Módulo de Abasto Cancún|
|Quintana Roo: Módulo de Abasto Cancún|Quintana Roo  |Módulo de Abasto Cancún|
|Quintana Roo: Módulo de Abasto Cancún|Quintana Roo  |Módulo de Abasto Cancún|
|Quintana Roo: Módulo de Abasto Cancún|Quintana Roo  |Módulo de Abasto Cancún|
|Quintana Roo: Módulo de Abasto Cancún|Quintana Roo  |Módulo de Abasto Cancún|
|Quintana Roo: Módulo de Abasto Cancún|Quintana Roo  |Módulo de Abasto Cancún|
|Quintana Roo: Mód

In [56]:
df_destinos_final.write \
    .mode("overwrite") \
    .partitionBy("anio", "mes") \
    .parquet("output/precios_particionado_fecha/")


print("¡Reporte guardado exitosamente en parquet!")

¡Reporte guardado exitosamente en parquet!


In [57]:
df_destinos_final.write \
    .mode("overwrite") \
    .partitionBy("Origen") \
    .parquet("output/precios_particionado_origen/")


print("¡Reporte por origenes guardado exitosamente en parquet!")

df_destinos_final.write \
    .mode("overwrite") \
    .partitionBy("estado_destino") \
    .parquet("output/precios_particionado_destino_estados/")


print("¡Reporte por destinos guardado exitosamente en parquet!")


¡Reporte por origenes guardado exitosamente en parquet!
¡Reporte por destinos guardado exitosamente en parquet!


In [34]:
from pyspark.sql.functions import col, avg, round

# Agrupamos por año y mes
# Calculamos el promedio del 'Precio Kg' 
# Redondeamos a 2 decimales para que sea formato moneda
# Ordenamos de menor a mayor para tener una línea de tiempo perfecta
df_promedios = df_limpio.groupBy("anio", "mes") \
    .agg(
        round(avg(col("Precio Kg")), 2).alias("Precio_Promedio_kg")
    ) \
    .orderBy("anio", "mes")

# Verificamos cómo quedó nuestra tabla resumen antes de guardarla
print("Vista previa del reporte:")
df_promedios.show(20)

Vista previa del reporte:
+----+---+------------------+
|anio|mes|Precio_Promedio_kg|
+----+---+------------------+
|2001|  1|             13.18|
|2001|  2|             13.02|
|2001|  3|             13.03|
|2001|  4|             13.35|
|2001|  5|             14.98|
|2001|  6|             15.19|
|2001|  7|             13.79|
|2001|  8|             12.95|
|2001|  9|             11.15|
|2001| 10|              9.27|
|2001| 11|              8.09|
|2001| 12|              7.51|
|2002|  1|              7.32|
|2002|  2|              7.24|
|2002|  3|              7.64|
|2002|  4|              8.13|
|2002|  5|              8.07|
|2002|  6|               9.0|
|2002|  7|             13.14|
|2002|  8|             13.44|
+----+---+------------------+
only showing top 20 rows



In [35]:
# 1. Creamos una lista en Python con todos los años y meses posibles (del 2001 al 2026)
años = range(2001, 2027)
meses = range(1, 13)

# Generamos todas las combinaciones posibles (año, mes)
combinaciones_ideales = [(a, m) for a in años for m in meses]

# 2. Convertimos esta lista a un DataFrame de Spark
df_calendario = spark.createDataFrame(combinaciones_ideales, ["anio_ideal", "mes_ideal"])

# 3. Cruzamos el calendario ideal con tus datos usando "left_anti"
meses_faltantes = df_calendario.join(
    df_promedios,
    (df_calendario.anio_ideal == df_promedios.anio) & (df_calendario.mes_ideal == df_promedios.mes),
    "left_anti"
).orderBy("anio_ideal", "mes_ideal")

# 4. Mostramos los meses que faltan
meses_faltantes.show(50)

+----------+---------+
|anio_ideal|mes_ideal|
+----------+---------+
|      2020|        6|
|      2026|        5|
|      2026|        6|
|      2026|        7|
|      2026|        8|
|      2026|        9|
|      2026|       10|
|      2026|       11|
|      2026|       12|
+----------+---------+



In [42]:
#Dado que los datos van hasta abril de 2026, solo faltaría el mes 6 del 2020, es decir, datos para Junio
#Como falta solo un dato, usamos el valor promedio entre mayo y julio para rellenar el mes faltante
from pyspark.sql.functions import col
import builtins # Importamos las funciones base de Python


# Extraemos el precio del mes anterior (Mayo 2020)
# Usamos collect()[0][0] para sacar el número exacto del DataFrame a una variable de Python
precio_mayo = df_promedios.filter((col("anio") == 2020) & (col("mes") == 5)) \
                          .select("Precio_Promedio_kg").collect()[0][0]

# Extraemos el precio del mes siguiente (Julio 2020)
precio_julio = df_promedios.filter((col("anio") == 2020) & (col("mes") == 7)) \
                           .select("Precio_Promedio_kg").collect()[0][0]

# Calculamos la interpolación (el promedio de ambos) redondeado a 2 decimales
precio_junio_imputado = builtins.round((precio_mayo + precio_julio) / 2, 2)
print(f"Precio calculado e imputado para Junio 2020: ${precio_junio_imputado}")

# 4. Creamos un nuevo DataFrame pequeño con esta única fila
nueva_fila = spark.createDataFrame(
    [(2020, 6, precio_junio_imputado)], 
    ["anio", "mes", "Precio_Promedio_kg"]
)

# 5. Lo unimos a nuestro DataFrame original y reordenamos para que quede en su lugar
df_promedios_completo = df_promedios.union(nueva_fila).orderBy("anio", "mes")

# Comprobamos visualmente que el año 2020 ahora esté completo
df_promedios_completo.filter(col("anio") == 2020).show()

Precio calculado e imputado para Junio 2020: $46.52
+----+---+------------------+
|anio|mes|Precio_Promedio_kg|
+----+---+------------------+
|2020|  1|             35.67|
|2020|  2|             35.67|
|2020|  3|             41.72|
|2020|  4|             45.38|
|2020|  5|             47.82|
|2020|  6|             46.52|
|2020|  7|             45.22|
|2020|  8|             46.45|
|2020|  9|              42.5|
|2020| 10|             39.46|
|2020| 11|              35.5|
|2020| 12|             32.02|
+----+---+------------------+



In [43]:
#Guardamos para posteriormente utilizar ARIMA para predicción de precios

ruta_salida_csv = "output/serie_de_tiempo_precios_mayoreo"

# Usamos coalesce(1) para forzar a Spark a juntar todo en un solo archivo .csv
df_promedios_completo.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(ruta_salida_csv)

print("¡Reporte de serie de tiempo guardado exitosamente en CSV!")

¡Reporte de serie de tiempo guardado exitosamente en CSV!


In [60]:
#Hallamos una serie de tiempo similar pero unicamente para La central de abasto de cdmx

df_cdmx = spark.read.parquet('output/precios_particionado_destino_estados').filter(F.col("estado_destino") == 'DF')

df_cdmx.show(10)

print(df_cdmx.columns)

+----------+--------------+---------+--------------------+----------+----------+-----------+--------------+---------+----+---+--------------------+--------------+
|     Fecha|  Presentacion|   Origen|             Destino|Precio Min|Precio Max|Precio Frec|Cantidad_Kilos|Precio Kg|anio|mes| centro_distribucion|estado_destino|
+----------+--------------+---------+--------------------+----------+----------+-----------+--------------+---------+----+---+--------------------+--------------+
|2001-01-05|Caja de 20 kg.|Michoacán|DF: Central de Ab...|     220.0|     240.0|      230.0|          20.0|     11.5|2001|  1|Central de Abasto...|            DF|
|2001-01-08|Caja de 20 kg.|Michoacán|DF: Central de Ab...|     210.0|     230.0|      220.0|          20.0|     11.0|2001|  1|Central de Abasto...|            DF|
|2001-01-09|Caja de 20 kg.|Michoacán|DF: Central de Ab...|     210.0|     230.0|      220.0|          20.0|     11.0|2001|  1|Central de Abasto...|            DF|
|2001-01-10|Caja de 20

In [61]:
df_cdmx.select('centro_distribucion').distinct().show(truncate=False)

+----------------------------------+
|centro_distribucion               |
+----------------------------------+
|Central de Abasto de Iztapalapa DF|
+----------------------------------+



In [67]:
from pyspark.sql.functions import col, avg, round

# Agrupamos por año y mes
# Calculamos el promedio del 'Precio Kg' 
# Redondeamos a 2 decimales para que sea formato moneda
# Ordenamos de menor a mayor para tener una línea de tiempo perfecta
df_nueve_kilos = df_cdmx.filter(col('Presentacion') == 'Caja de 9 kg.')

df_nueve_kilos.show(5)
print(f'La cantidad de registros con la caja de 9 kilos son: {df_nueve_kilos.count()}')

+----------+-------------+---------+--------------------+----------+----------+-----------+--------------+-----------------+----+---+--------------------+--------------+
|     Fecha| Presentacion|   Origen|             Destino|Precio Min|Precio Max|Precio Frec|Cantidad_Kilos|        Precio Kg|anio|mes| centro_distribucion|estado_destino|
+----------+-------------+---------+--------------------+----------+----------+-----------+--------------+-----------------+----+---+--------------------+--------------+
|2011-07-04|Caja de 9 kg.|Michoacán|DF: Central de Ab...|     450.0|     490.0|      470.0|           9.0|52.22222222222222|2011|  7|Central de Abasto...|            DF|
|2011-07-05|Caja de 9 kg.|Michoacán|DF: Central de Ab...|     450.0|     480.0|      470.0|           9.0|52.22222222222222|2011|  7|Central de Abasto...|            DF|
|2011-07-06|Caja de 9 kg.|Michoacán|DF: Central de Ab...|     430.0|     460.0|      450.0|           9.0|             50.0|2011|  7|Central de Abasto

In [68]:
df_promedios_9 = df_nueve_kilos.groupBy("anio", "mes") \
    .agg(
        round(avg(col("Precio Frec")), 2).alias("Precio_Promedio_kg")
    ) \
    .orderBy("anio", "mes")

# Verificamos cómo quedó nuestra tabla resumen antes de guardarla
print("Reporte por mes y año para el precio por una caja de 9kg:")
df_promedios_9.show(20)


Reporte por mes y año para el precio por una caja de 9kg:
+----+---+------------------+
|anio|mes|Precio_Promedio_kg|
+----+---+------------------+
|2011|  7|            463.13|
|2011|  8|            307.65|
|2011|  9|            216.67|
|2011| 10|            164.44|
|2011| 11|            160.56|
|2011| 12|            164.09|
|2012|  1|             179.0|
|2012|  2|            177.37|
|2012|  3|             194.5|
|2012|  4|            223.33|
|2012|  5|             256.0|
|2012|  6|            264.76|
|2012|  7|             241.0|
|2012|  8|            233.48|
|2012|  9|            165.79|
|2012| 10|            142.27|
|2012| 11|            124.44|
|2012| 12|             119.0|
|2013|  1|            120.91|
|2013|  2|            107.37|
+----+---+------------------+
only showing top 20 rows



In [71]:
#observamos los años que tenemos 
df_promedios_9.select('anio').distinct().orderBy('anio').show()

+----+
|anio|
+----+
|2011|
|2012|
|2013|
|2014|
|2017|
|2018|
|2019|
|2020|
|2021|
|2022|
|2023|
|2024|
|2025|
|2026|
+----+



In [73]:
#Tomaremos solo del año 2018 en adelante y observaremos cuántos meses hacen falta
df_promedios_9_red = df_promedios_9.filter(col('anio') > 2017)

# 1. Creamos una lista en Python con todos los años y meses posibles (del 2001 al 2026)
años = range(2018, 2027)
meses = range(1, 13)

# Generamos todas las combinaciones posibles (año, mes)
combinaciones_ideales = [(a, m) for a in años for m in meses]

# 2. Convertimos esta lista a un DataFrame de Spark
df_calendario = spark.createDataFrame(combinaciones_ideales, ["anio_ideal", "mes_ideal"])

# 3. Cruzamos el calendario ideal con tus datos usando "left_anti"
meses_faltantes = df_calendario.join(
    df_promedios_9_red,
    (df_calendario.anio_ideal == df_promedios_9_red.anio) & (df_calendario.mes_ideal == df_promedios_9_red.mes),
    "left_anti"
).orderBy("anio_ideal", "mes_ideal")

# 4. Mostramos los meses que faltan
meses_faltantes.show(50)

+----------+---------+
|anio_ideal|mes_ideal|
+----------+---------+
|      2020|        6|
|      2021|        1|
|      2022|        6|
|      2026|        5|
|      2026|        6|
|      2026|        7|
|      2026|        8|
|      2026|        9|
|      2026|       10|
|      2026|       11|
|      2026|       12|
+----------+---------+



In [79]:
from pyspark.sql.functions import col, count, when

df_serie = df_calendario.join(
    df_promedios_9_red,
    (df_calendario.anio_ideal == df_promedios_9_red.anio) & (df_calendario.mes_ideal == df_promedios_9_red.mes),
    "left"
).select(
    col("anio_ideal").alias("anio"),
    col("mes_ideal").alias("mes"),
    col("Precio_Promedio_kg")
)


df_serie.select([count(when(col(c).isNull(), c)).alias(c) for c in df_serie.columns]).show()


+----+---+------------------+
|anio|mes|Precio_Promedio_kg|
+----+---+------------------+
|   0|  0|                11|
+----+---+------------------+



In [88]:
import pandas as pd

# 1. Convertimos tu DataFrame de Spark (con el cruce del calendario ideal) a Pandas
df_pandas = df_serie.orderBy("anio", "mes").toPandas()

# 2. Eliminamos los "nulos falsos" del futuro
# La virgulilla (~) significa "NOT" en Pandas. Es decir, quédate con todo EXCEPTO esto:
df_pandas = df_pandas[~((df_pandas['anio'] == 2026) & (df_pandas['mes'] >= 5))]

# 3. Aplicamos la interpolación lineal para rellenar 2020, 2021 y 2022
df_pandas['Precio_Promedio_kg'] = df_pandas['Precio_Promedio_kg'].interpolate(method='linear')

df_pandas.isna().sum()

anio                  0
mes                   0
Precio_Promedio_kg    0
dtype: int64

In [91]:
# Guarda dentro de la carpeta 'output'
df_pandas.to_csv("output/precios_cdmx_caja_9kg_2018_2026.csv", index=False, encoding="utf-8-sig")

In [92]:
spark.stop()